# `sr_gauss_all_test` — Interactive PySR Run

A scaled-down version of the `sr_gauss_all` production run for interactive exploration.
Uses the same inputs, operators, and loss function, but with fewer samples, populations,
and a short timeout so it returns equations in seconds rather than hours.

| Parameter | Production | Test |
|-----------|-----------|------|
| `subsetsize` | 10,000 | 500 |
| `populations` | 50 | 10 |
| `populationsize` | 150 | 30 |
| `targettotal` | 500,000 | 2,000 |
| `procs` | 50 | 4 |
| `timeout` | 5.5 hr | 120 s |

In [7]:
import os
import sys
import copy
import tempfile
import shutil
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd

from scripts.utils import Config
from scripts.models.sr.train import load_data, subsample, fit, save, select_pareto_elbow

## Configuration

Build a scaled-down copy of the `sr_gauss_all` run config.

In [12]:
config  = Config()
sr      = copy.deepcopy(config.sr)

RUNNAME    = 'sr_gauss_all_test'
RUNCONFIG  = sr['runs']['sr_gauss_all']          # same inputs as production
FIELDVARS  = RUNCONFIG['fieldvars']              # ['rh', 'thetae', 'thetaestar']
LOCALVARS  = RUNCONFIG['localvars']              # ['lf', 'lhf', 'shf']
PREDICTORS = FIELDVARS + LOCALVARS

# --- scaled-down search parameters ---
SUBSETSIZE = 500
PROCS      = 4
TIMEOUT    = 120       # seconds

sr['searchparams']['populations']       = 10
sr['searchparams']['populationsize']    = 30
sr['searchparams']['targettotal']       = 2000
# sr['searchparams'].pop('iterations', None)   # targettotal takes precedence
sr['searchparams']['batchsize']         = 50

sp       = sr['searchparams']
popcount = sp['populations']
iterseff = sp['targettotal'] // popcount

print(f'Predictors : {PREDICTORS}')
print(f'Populations: {popcount}  |  Iters/population: {iterseff}  |  Workers: {PROCS}  |  Timeout: {TIMEOUT}s')

Predictors : ['rh', 'thetae', 'thetaestar', 'lf', 'lhf', 'shf']
Populations: 10  |  Iters/population: 200  |  Workers: 4  |  Timeout: 120s


## Load Data

Load normalized train and validation splits, kernel-integrate the 3-D field variables
using the learned `param_gauss_all` weights (averaged across seeds), then concatenate
the scalar local variables. This exactly mirrors the production `load_data` call.

In [9]:
print('Loading train split...')
xtrain, ytrain, refdatrain, vmtrain = load_data('train', RUNCONFIG, config, time_offset=0)

print('Loading valid split...')
xvalid, yvalid, _, vmvalid = load_data('valid', RUNCONFIG, config,
                                        time_offset=int(refdatrain.sizes['time']))

xfit = pd.concat([xtrain[vmtrain], xvalid[vmvalid]]).reset_index(drop=True)
yfit = np.concatenate([ytrain[vmtrain], yvalid[vmvalid]])
del xtrain, xvalid, ytrain, yvalid, refdatrain

print(f'Combined finite samples: {len(xfit):,}')
xfit.describe()

Loading train split...
Loading valid split...
Combined finite samples: 8,624,448


,rh,thetae,thetaestar,lf,lhf,shf,timeidx
count,8.624448e+06,8.624448e+06,8.624448e+06,8.624448e+06,8.624448e+06,8.624448e+06,8.624448e+06
mean,-1.427015e-01,6.906726e-01,2.908896e-02,2.925903e-01,6.874994e-03,2.284334e-03,6.623500e+03
std,7.894574e-01,5.377156e-01,6.423157e-01,4.478272e-01,9.888903e-01,9.786609e-01,3.824368e+03
min,-2.941444e+00,-2.116019e+00,-1.223903e+00,0.000000e+00,-8.890768e+00,-1.240867e+01,0.000000e+00
25%,-7.195634e-01,3.094557e-01,-3.839337e-01,0.000000e+00,-6.113117e-01,2.567759e-02,3.311750e+03
50%,-5.083379e-02,6.194977e-01,-1.352963e-01,0.000000e+00,-1.335129e-02,2.182616e-01,6.623500e+03
75%,4.935950e-01,1.040777e+00,2.194438e-01,9.519936e-01,6.967799e-01,3.533137e-01,9.935250e+03
max,1.452909e+00,3.577948e+00,5.867566e+00,1.000000e+00,2.208292e+00,3.941637e+00,1.324700e+04


## Subsample

Draw ~500 samples with proportional coverage of the precipitation distribution
(one-decade log10 bins from 0.0001–100 mm, plus a dry bin).

In [10]:
xsub, ysub = subsample(xfit, yfit, subsetsize=SUBSETSIZE, seed=sr['seed'])
del xfit, yfit

print(f'Subset size: {len(xsub):,} samples  ×  {xsub.shape[1]} predictors')
xsub.describe()

Subset size: 1,953 samples  ×  6 predictors


,rh,thetae,thetaestar,lf,lhf,shf
count,1953.000000,1953.000000,1953.000000,1953.000000,1953.000000,1953.000000
mean,-0.201191,0.481957,-0.148040,0.292590,-0.074193,-0.324414
std,0.890196,0.475143,0.476842,0.441056,0.975017,1.326201
min,-2.429338,-0.936871,-1.082314,0.000000,-3.830805,-8.966317
25%,-0.784827,0.177328,-0.450982,0.000000,-0.522844,-0.050200
50%,-0.134276,0.438817,-0.284977,0.000000,-0.015333,0.157642
75%,0.547343,0.740144,0.010240,0.951994,0.455189,0.283454
max,1.391908,2.120970,2.377098,1.000000,1.819328,0.864516


## Fit PySR

Runs the symbolic regression search. With the test parameters this typically
finishes in under 2 minutes. Julia startup adds ~1–2 minutes on first run.

In [ ]:
tmpdir = tempfile.mkdtemp(prefix='pysr_test_')
try:
    model = fit(xsub, ysub, PREDICTORS, sr, PROCS, TIMEOUT, tmpdir)
finally:
    shutil.rmtree(tmpdir, ignore_errors=True)

print('Search complete.')

Compiling Julia backend...
16:01:49 - INFO - Compiling Julia backend...
[ Info: Automatically setting `--heap-size-hint=50446M` on each Julia process. You can configure this with the `heap_size_hint_in_bytes` parameter.


## Pareto Frontier

All discovered equations sorted by complexity, with the elbow selection highlighted.

In [ ]:
equations = model.equations_.sort_values('complexity').reset_index(drop=True)
best      = select_pareto_elbow(equations)

display_cols = ['complexity', 'loss', 'equation']
styled = equations[display_cols].style.apply(
    lambda row: ['background-color: #d4edda' if row.name == best.name else '' for _ in row],
    axis=1
)
display(styled)

In [ ]:
print('=== Elbow-selected equation ===')
print(f'  {best["equation"]}')
print(f'  Complexity : {best["complexity"]}')
print(f'  Train loss : {best["loss"]:.6f}')

## Save (optional)

Uncomment to persist the test model to `models/sr/` alongside the production runs.
The output files will be named `sr_gauss_all_test_pareto.pkl` and `sr_gauss_all_test_equations.csv`.

In [ ]:
# save(model, RUNNAME, config)